# Hospedando crew multi-agente CrewAI com modelos do Amazon Bedrock no Amazon Bedrock AgentCore Runtime

## Visão Geral

Neste tutorial, aprenderemos como hospedar seu crew multi-agente existente usando o Amazon Bedrock AgentCore Runtime. 

Focaremos em um exemplo de CrewAI com modelo do Amazon Bedrock. Para Strands Agents com modelo do Amazon Bedrock, consulte [aqui](../01-strands-with-bedrock-model) e para Strands Agents com um modelo OpenAI, consulte [aqui](../03-strands-with-openai-model).


### Detalhes do Tutorial

| Informação | Detalhes |
|:--------------------|:-----------------------------------------------------------------------------|
| Tipo de tutorial | Conversacional |
| Tipo de agente | Crew multi-agente |
| Framework Agêntico | CrewAI |
| Modelo LLM | Anthropic Claude Haiku 4.5 |
| Componentes | Hospedagem de agente no AgentCore Runtime. Usando CrewAI e Modelo Amazon Bedrock |
| Vertical do tutorial | Multisetorial |
| Complexidade | Fácil |
| SDK utilizado | Amazon BedrockAgentCore Python SDK e boto3 |

### Arquitetura do Tutorial

Neste tutorial, descreveremos como implantar um crew multi-agente existente no AgentCore runtime. 

Para fins de demonstração, usaremos um crew CrewAI usando modelos do Amazon Bedrock

Em nosso exemplo, usaremos um crew de pesquisa com dois agentes: um pesquisador e um analista.
<div style="text-align:left">
    <img src="images/architecture_runtime.png" width="60%"/>
</div>


### Principais Recursos do Tutorial

* Hospedando Agentes no Amazon Bedrock AgentCore Runtime
* Usando modelos do Amazon Bedrock
* Usando CrewAI

## Pré-requisitos

Para executar este tutorial, você precisará de:
* Python 3.10+
* Gerenciador de pacotes uv
* Credenciais AWS
* Docker em execução

Além disso, precisamos instalar algumas dependências: 
* Amazon Bedrock AgentCore SDK
* CrewAI 
* Langchain community package
* Duckduckgo search

Empacotamos todas as dependências necessárias em um arquivo pyproject.toml para que possam ser instaladas convenientemente. 

In [ ]:
!uv sync --active --force-reinstall

## Criando seu crew multi-agente e experimentando localmente

Antes de implantar nossos agentes no AgentCore Runtime, vamos desenvolvê-los e executá-los localmente para fins de experimentação.

In this guide, we’ll walk through creating a research crew that will help us research and analyze a topic, then create a comprehensive report. This practical example demonstrates how AI agents can collaborate to accomplish complex tasks. The example is adapted from a [getting started guide](https://docs.crewai.com/en/guides/crews/first-crew) provided directly by CrewAI.

A arquitetura local é a seguinte:

<div style="text-align:left">
    <img src="images/architecture_local.png" width="60%"/>
</div>


### Definindo agentes, tarefas e crew

Primeiro criaremos os artefatos que definem um agente CrewAI local, incluindo: 
* agents.yaml, definindo os dois agentes envolvidos em nosso crew
* tasks.yaml, definindo as tarefas a serem executadas pelos agentes em nosso crew
* crew.py, definindo nosso crew consistindo de agentes trabalhando em tarefas conforme definido
* main.py, nosso ponto de entrada local que inicia a execução do crew

In [ ]:
import os
os.makedirs('research_crew/config', exist_ok=True)

In [ ]:
%%writefile research_crew/config/agents.yaml
researcher:
  role: >
    Senior Research Specialist for {topic}
  goal: >
    Find comprehensive and accurate information about {topic}
    with a focus on recent developments and key insights
  backstory: >
    You are an experienced research specialist with a talent for
    finding relevant information from various sources. You excel at
    organizing information in a clear and structured manner, making
    complex topics accessible to others.
  llm: bedrock/global.anthropic.claude-haiku-4-5-20251001-v1:0

analyst:
  role: >
    Data Analyst and Report Writer for {topic}
  goal: >
    Analyze research findings and create a comprehensive, well-structured
    report that presents insights in a clear and engaging way
  backstory: >
    You are a skilled analyst with a background in data interpretation
    and technical writing. You have a talent for identifying patterns
    and extracting meaningful insights from research data, then
    communicating those insights effectively through well-crafted reports.
  llm: bedrock/global.anthropic.claude-haiku-4-5-20251001-v1:0

In [ ]:
%%writefile research_crew/config/tasks.yaml
research_task:
  description: >
    Conduct thorough research on {topic}. Focus on:
    1. Key concepts and definitions
    2. Historical development and recent trends
    3. Major challenges and opportunities
    4. Notable applications or case studies
    5. Future outlook and potential developments

    Make sure to organize your findings in a structured format with clear sections.
  expected_output: >
    A comprehensive research document with well-organized sections covering
    all the requested aspects of {topic}. Include specific facts, figures,
    and examples where relevant.
  agent: researcher

analysis_task:
  description: >
    Analyze the research findings and create a comprehensive report on {topic}.
    Your report should:
    1. State the topic and begin with an executive summary
    2. Include all key information from the research
    3. Provide insightful analysis of trends and patterns
    4. Offer recommendations or future considerations
    5. Be formatted in a professional, easy-to-read style with clear headings
  expected_output: >
    A polished, professional report on {topic} that presents the research
    findings with added analysis and insights. The report should be well-structured
    with an executive summary, main sections, and conclusion.
  agent: analyst
  context:
    - research_task

In [ ]:
%%writefile research_crew/crew.py
from crewai import Agent, Crew, Process, Task
from crewai.project import CrewBase, agent, crew, task
from crewai.agents.agent_builder.base_agent import BaseAgent
from typing import List
from langchain_community.tools import DuckDuckGoSearchRun
from crewai.tools import BaseTool
from crewai_tools import SerperDevTool
from pydantic import Field


class SearchTool(BaseTool):
     name: str = "Search"
     description: str = "Useful for searching the web for information."
     search: DuckDuckGoSearchRun = Field(default_factory=DuckDuckGoSearchRun)

     def _run(self, query: str) -> str:
         """Execute the search query and return results"""
         try:
             return self.search.invoke(query)
         except Exception as e:
             return f"Error performing search: {str(e)}"

@CrewBase
class ResearchCrew():
    """Research crew for comprehensive topic analysis and reporting"""

    agents: List[BaseAgent]
    tasks: List[Task]

    @agent
    def researcher(self) -> Agent:
        return Agent(
            config=self.agents_config['researcher'], # type: ignore[index]
            verbose=True,
            tools=[
                #SerperDevTool()
                SearchTool()
                ]
        )

    @agent
    def analyst(self) -> Agent:
        return Agent(
            config=self.agents_config['analyst'], # type: ignore[index]
            verbose=True
        )

    @task
    def research_task(self) -> Task:
        return Task(
            config=self.tasks_config['research_task'] # type: ignore[index]
        )

    @task
    def analysis_task(self) -> Task:
        return Task(
            config=self.tasks_config['analysis_task'], # type: ignore[index]
            #output_file='output/report.md'
        )

    @crew
    def crew(self) -> Crew:
        """Creates the research crew"""
        return Crew(
            agents=self.agents,
            tasks=self.tasks,
            process=Process.sequential,
            verbose=True,
        )

In [ ]:
%%writefile research_crew/main.py
import os
from research_crew.crew import ResearchCrew

# Create output directory if it doesn't exist
os.makedirs('output', exist_ok=True)

def run():
    """
    Run the research crew.
    """
    inputs = {
        'topic': 'Artificial Intelligence in Healthcare'
    }

    # Create and run the crew
    result = ResearchCrew().crew().kickoff(inputs=inputs)

    # Print the result
    print("\n\n=== FINAL REPORT ===\n\n")
    print(result.raw)


if __name__ == "__main__":
    run()

### Invocando crew localmente

Finalmente, podemos usar a CLI do CrewAI para iniciar o crew localmente. Alternativamente, também poderíamos simplesmente executar nosso ponto de entrada local main.py. Isso pode levar alguns minutos. 

In [ ]:
!crewai run

## Implantando crew multi-agente no Amazon Bedrock AgentCore

Para aplicações agênticas de nível de produção, precisaremos executar nosso crew na nuvem. Portanto, implantaremos nosso crew no Amazon Bedrock AgentCore. 

A arquitetura aqui será a seguinte:

<div style="text-align:left">
     <img src="images/architecture_local.png" width="60%"/>
</div>

Implantar o crew no AgentCore envolve as seguintes etapas: 

### Ponto de entrada remoto

Primeiro, criamos um ponto de entrada remoto. Com o AgentCore Runtime, decoraremos a parte de invocação do nosso agente com o decorador @app.entrypoint e o teremos como ponto de entrada para nosso runtime. Isso também envolve: 
* Importar o Runtime App com `from bedrock_agentcore.runtime import BedrockAgentCoreApp`
* Inicializar o App em nosso código com `app = BedrockAgentCoreApp()`
* Decorar a função de invocação com o decorador `@app.entrypoint`
* Deixar o AgentCoreRuntime controlar a execução do agente com `app.run()`

### O que acontece nos bastidores?

Quando você usa `BedrockAgentCoreApp`, ele automaticamente:

* Cria um servidor HTTP que escuta na porta 8080
* Implementa o endpoint `/invocations` necessário para processar os requisitos do agente
* Implementa o endpoint `/ping` para verificações de integridade (muito importante para agentes assíncronos)
* Gerencia tipos de conteúdo e formatos de resposta adequados
* Gerencia tratamento de erros de acordo com os padrões AWS                                                                                                                                                                        

In [ ]:
%%writefile research_crew/research_crew.py
import os
from research_crew.crew import ResearchCrew

# ---------- Agentcore imports --------------------
from bedrock_agentcore.runtime import BedrockAgentCoreApp

app = BedrockAgentCoreApp()
#------------------------------------------------


@app.entrypoint
def agent_invocation(payload, context):
    """Handler for agent invocation"""
    print(f'Payload: {payload}')
    try: 
        # Extract user message from payload with default
        user_message = payload.get("prompt", "Artificial Intelligence in Healthcare")
        print(f"Processing topic: {user_message}")
        
        # Create crew instance and run synchronously
        research_crew_instance = ResearchCrew()
        crew = research_crew_instance.crew()
        
        # Use synchronous kickoff instead of async - this avoids all event loop issues
        result = crew.kickoff(inputs={'topic': user_message})

        print("Context:\n-------\n", context)
        print("Result Raw:\n*******\n", result.raw)
        
        # Safely access json_dict if it exists
        if hasattr(result, 'json_dict'):
            print("Result JSON:\n*******\n", result.json_dict)
        
        return {"result": result.raw}
        
    except Exception as e:
        print(f'Exception occurred: {e}')
        return {"error": f"An error occurred: {str(e)}"}

if __name__ == "__main__":
    app.run()

### Implantando o agente no AgentCore Runtime

A operação `CreateAgentRuntime` suporta opções abrangentes de configuração, permitindo especificar imagens de contêiner, variáveis de ambiente e configurações de criptografia. Você também pode configurar configurações de protocolo (HTTP, MCP) e mecanismos de autorização para controlar como seus clientes se comunicam com o agente. 

**Nota:** A melhor prática de operações é empacotar o código como contêiner e enviar para o ECR usando pipelines CI/CD e IaC

Neste tutorial, usaremos o Amazon Bedrock AgentCode Python SDK para empacotar facilmente seus artefatos e implantá-los no AgentCore runtime.

#### Configurar implantação do AgentCore Runtime

Primeiro, usaremos nosso kit inicial para configurar a implantação do AgentCore Runtime com um ponto de entrada, a função de execução que acabamos de criar e um arquivo de requisitos. Também configuraremos o kit inicial para criar automaticamente o repositório Amazon ECR no lançamento.

O configure do AgentCore é necessário para gerar um Dockerfile contendo um blueprint para o contêiner Docker no qual a carga de trabalho será executada e um .bedrock_agentcore.yaml contendo a configuração da carga de trabalho agêntica. Durante a etapa de configuração, seu arquivo docker será gerado com base no código do seu aplicativo.

<div style="text-align:left">
    <img src="images/configure.png" width="60%"/>
</div>

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "research_crew_getting_started"
response = agentcore_runtime.configure(
    entrypoint="research_crew/research_crew.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    region=region,
    agent_name=agent_name
)
response

#### Lançando agente no AgentCore Runtime: implantando a carga de trabalho agêntica remota

Agora que temos um arquivo docker, vamos lançar o agente no AgentCore Runtime. Isso criará o repositório Amazon ECR e o AgentCore Runtime. O launch do AgentCore então implantará a carga de trabalho agêntica na nuvem. Isso inclui criar uma imagem Docker e enviá-la para o ECR, bem como preparar um endpoint para uso.


<div style="text-align:left">
    <img src="images/launch.png" width="85%"/>
</div>

In [ ]:
launch_result = agentcore_runtime.launch()

#### Verificando o Status do AgentCore Runtime

Agora que implantamos o AgentCore Runtime, vamos verificar seu status de implantação

In [ ]:
import time
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
status

### Invocando AgentCore Runtime com boto3

Agora que seu AgentCore Runtime foi criado, você pode invocá-lo com qualquer AWS SDK. Por exemplo, você pode usar o método `invoke_agent_runtime` do boto3 para isso. Como este é um agente de longa duração, estamos sobrescrevendo os valores padrão de `retries`, `connect_timeout` e `read_timeout`.

<div style="text-align:left">
    <img src="images/invoke.png" width=85%"/>
</div>

In [ ]:
from botocore.config import Config

# Configure retries and timeout
config = Config(
    retries={
        'max_attempts': 10,  # Increase max retries to 10 (default is 4)
        'mode': 'adaptive'   # Options: 'legacy', 'standard', 'adaptive'
    },
    connect_timeout=600,      # Connection timeout in seconds (default is 60)
    read_timeout=3000         # Read timeout in seconds (default is 60)
)

In [ ]:
import boto3
import json
from IPython.display import Markdown, display
agent_arn = launch_result.agent_arn
agentcore_client = boto3.client(
    'bedrock-agentcore',
    region_name=region
)

boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "What is 2+2?"})
)

# Capture the runtime session ID for lifecycle management
runtime_session_id = boto3_response.get('runtimeSessionId')
print(f"Runtime Session ID: {runtime_session_id}")

if "text/event-stream" in boto3_response.get("contentType", ""):
    content = []
    for line in boto3_response["response"].iter_lines(chunk_size=1):
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: "):
                line = line[6:]
                print(line)
                content.append(line)
    display(Markdown("\n".join(content)))
else:
    try:
        events = []
        for event in boto3_response.get("response", []):
            events.append(event)
    except Exception as e:
        events = [f"Error reading EventStream: {e}"]
    result = json.loads(events[0].decode("utf-8")) if events else "No response"
    display(Markdown(result if isinstance(result, str) else json.dumps(result, indent=2)))

### Parando uma Sessão

Você vai querer parar sessões individuais quando elas não forem mais necessárias.
Isso libera os recursos de microVM para essa sessão enquanto mantém o runtime ativo
para novas sessões. Abaixo demonstramos `stop_runtime_session`.

In [ ]:
# --- Inline Session Lifecycle Demo ---
# stop_runtime_session releases the microVM resources for this specific session while keeping the runtime alive for new sessions.

if runtime_session_id:
    agentcore_client.stop_runtime_session(
        agentRuntimeArn=agent_arn,
        runtimeSessionId=runtime_session_id,
        qualifier='DEFAULT'
    )
    print(f"✅ Session '{runtime_session_id}' stopped — microVM resources released")
else:
    print("⚠️ No session ID available to stop")

### Demonstração de Configuração de Ciclo de Vida (Ativa)

Agora vamos demonstrar como configurar um runtime com um tempo limite de inatividade mais curto.
Criaremos um segundo runtime com um tempo limite de inatividade de 5 minutos (300 segundos) para mostrar
como a configuração de ciclo de vida afeta o comportamento da sessão. Ambos os runtimes coexistirão.

In [ ]:
# --- Lifecycle Configuration Demo ---
# In production, choose a timeout appropriate for your workload:
#   - Development/testing: 5-15 minutes
#   - Interactive sessions: 30-60 minutes
#   - Long-running workloads: adjust as needed
#

agentcore_runtime_short = Runtime()
agent_name_short = "crewai_claude_short_timeout"

# Configure with shorter idle timeout
response_short = agentcore_runtime_short.configure(
    entrypoint="research_crew/research_crew.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="research_crew/requirements.txt",
    region=region,
    agent_name=agent_name_short
)

# Launch the second runtime
launch_result_short = agentcore_runtime_short.launch()
print(f"Second runtime launched: {launch_result_short.agent_id}")

# Wait for it to be ready
status_response_short = agentcore_runtime_short.status()
status_short = status_response_short.endpoint['status']
while status_short not in ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']:
    time.sleep(10)
    status_response_short = agentcore_runtime_short.status()
    status_short = status_response_short.endpoint['status']
    print(f"Short timeout runtime status: {status_short}")

# Now update the runtime with shorter idle timeout using boto3
# UpdateAgentRuntime is a full-replacement API — we must re-supply all required fields.
# First, retrieve the current runtime configuration.
agentcore_control_client = boto3.client('bedrock-agentcore-control', region_name=region)
current_runtime = agentcore_control_client.get_agent_runtime(
    agentRuntimeId=launch_result_short.agent_id
)

update_response = agentcore_control_client.update_agent_runtime(
    agentRuntimeId=launch_result_short.agent_id,
    agentRuntimeArtifact=current_runtime['agentRuntimeArtifact'],
    roleArn=current_runtime['roleArn'],
    networkConfiguration=current_runtime['networkConfiguration'],
    lifecycleConfiguration={
        'idleRuntimeSessionTimeout': 300  # 5 minutes
    }
)
print(f"✅ Runtime updated with 5-minute idle timeout")

# Invoke the second runtime to verify it works
invoke_response_short = agentcore_runtime_short.invoke({"prompt": "What is 3+3?"})
print(f"Second runtime response: {invoke_response_short['response'][0]}")

## Limpeza

Vamos agora limpar o AgentCore Runtime e recursos associados. Deletamos o runtime primeiro para evitar custos indesejados, depois limpamos recursos de suporte como repositórios ECR.

In [ ]:
launch_result.ecr_uri, launch_result.agent_id, launch_result.ecr_uri.split('/')[1]

In [ ]:
# --- Stop active sessions to release microVM resources ---
import boto3

agentcore_client = boto3.client('bedrock-agentcore', region_name=region)
agentcore_control_client = boto3.client('bedrock-agentcore-control', region_name=region)
ecr_client = boto3.client('ecr', region_name=region)

# Stop the active session to release its microVM resources
# In production, this is how you end individual user sessions while keeping the runtime alive
# AgentCore Runtime costs are based on vCPU and Memory — stopping sessions avoids undesired costs
# Note: If the session was already stopped in the earlier demo cell, this will raise a
# ResourceNotFoundException — the except block handles that gracefully.
if 'runtime_session_id' in locals() and runtime_session_id:
    try:
        agentcore_client.stop_runtime_session(
            agentRuntimeArn=launch_result.agent_arn,
            runtimeSessionId=runtime_session_id,
            qualifier='DEFAULT'
        )
        print(f"✅ Session '{runtime_session_id}' stopped")
    except Exception as e:
        print(f"⚠️ Failed to stop session '{runtime_session_id}': {e}")

# --- Delete both runtimes ---
# Original runtime
try:
    agentcore_control_client.delete_agent_runtime(
        agentRuntimeId=launch_result.agent_id,
    )
    print(f"✅ Original runtime '{launch_result.agent_id}' deleted")
except Exception as e:
    print(f"⚠️ Failed to delete original runtime: {e}")

# Short-timeout runtime
if 'launch_result_short' in locals():
    try:
        agentcore_control_client.delete_agent_runtime(
            agentRuntimeId=launch_result_short.agent_id,
        )
        print(f"✅ Short-timeout runtime '{launch_result_short.agent_id}' deleted")
    except Exception as e:
        print(f"⚠️ Failed to delete short-timeout runtime: {e}")

# --- Delete ECR repositories ---
try:
    ecr_client.delete_repository(
        repositoryName=launch_result.ecr_uri.split('/')[1],
        force=True
    )
    print(f"✅ ECR repository '{launch_result.ecr_uri.split('/')[1]}' deleted")
except Exception as e:
    print(f"⚠️ Failed to delete ECR repository: {e}")

if 'launch_result_short' in locals():
    try:
        ecr_client.delete_repository(
            repositoryName=launch_result_short.ecr_uri.split('/')[1],
            force=True
        )
        print(f"✅ Second ECR repository '{launch_result_short.ecr_uri.split('/')[1]}' deleted")
    except Exception as e:
        print(f"⚠️ Failed to delete second ECR repository: {e}")

## Parabéns!